For the Dataset.csv

In [ ]:
# !pip install torch==1.13.1 --extra-index-url https://download.pytorch.org/whl/cpu

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("Python version:", __import__("sys").version)

PyTorch version: 2.11.0+cpu
Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]


In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F
import gym
from gym import spaces
import os
import math
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from gym import spaces
import matplotlib.pyplot as plt
from tqdm import tqdm
from torch.optim import AdamW
import multiprocessing as mp
from torch.nn import Module
import warnings

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

# Replace 'your_excel_file.xlsx' with the actual path to your Excel file
excel_file_path = '/content/drive/MyDrive/Open Port Risk Assessment/dataset.xlsx'

# Replace 'column_name' with the actual name of the column you want to fetch
RunTime = 'Run Time'

# Read the Excel file into a pandas DataFrame
df = pd.read_excel(excel_file_path)

# Fetch the specified column and convert it to a Python array
RunTime = df[RunTime].to_numpy()

# Print or use the array as needed
# print(RunTime)


In [ ]:
print("Dataset loaded successfully")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Run Time samples:", RunTime[:5])

Dataset loaded successfully
Shape: (1000, 18)
Columns: ['Job Number', 'Submit Time', 'Wait Time', 'Run Time', 'Number of Allocated Processors', 'Average CPU Time Used', 'Used Memory', 'Requested Number of Processors', 'Requested Time', 'Requested Memory', 'Status', 'User ID', 'Group ID', 'Executable (Application) Number', 'Queue Number', 'Partition Number', 'Preceding Job Number', 'Think Time from Preceding Job']
Run Time samples: [14400 14400    45   154    50]


In [ ]:
start_index = 0
end_index = 999
task_lengths = RunTime[start_index:end_index]
print(task_lengths)

[ 14400  14400     45    154     50   1112 176777     45    353     42
   2114 172800     49     42     53     42    657    180    365    354
    374     31    327  25210  41846  42111  42240  42488  42341  42114
  42891    109  41534  41866  41994  39720  41951  43726  43788  39474
  39751  40137  43203  39377  39516  40033    347    347    325    313
    346    313    346    335    313    313    342  95052    342     31
    907    296     31    335     33    326     62     43     53     54
     53    338    332    341     53     43     90     86    681     85
     84     93     95     93     92    197    104    467    190    166
    212    214    271   7297    888   3452   3669  21507   6610   1743
   1823    157   3780   2162    310   2694    395    385     91    367
    987    135     42    223   1348    178    658    302   1384     87
    155    332    321    409    294    590    325   1242  11296   1059
    185    707    610  43190  43157  43857   3949  43382  43623  43706
     9

In [ ]:
# ENVIRONMENT

class Datacenter:
    def __init__(self, name, location, memory, cost_per_hour, cost_per_memory, bandwidth):
        self.name = name
        self.location = location
        self.memory = memory
        self.cost_per_hour = cost_per_hour
        self.cost_per_memory = cost_per_memory
        self.bandwidth = bandwidth
        self.hosts = []

    def add_host(self, host):
        self.hosts.append(host)

    def get_hosts(self):
        return self.hosts

    def show_hosts(self):
        print(self.hosts)


# Processing Element Types
PE_Type_list = {
    '1': {'MIPS': 100},
    '2': {'MIPS': 50},
    '3': {'MIPS': 10}
}


class Host:
    def __init__(self, name, Datacenter, cpu_cores, memory, ram, pe_type, bandwidth):
        self.name = name
        self.Datacenter = Datacenter
        self.cpu_cores = cpu_cores
        self.memory = memory
        self.ram = ram
        self.pe_type = pe_type
        self.bandwidth = bandwidth
        self.vms = []

    def add_vm(self, vm):
        self.vms.append(vm)

    def get_mips(self, pe_type):
        return PE_Type_list[str(pe_type)]['MIPS']

    def get_vms(self):
        return self.vms


# VM Types
VM_type = {
    'Type_1': {
        'Task_Memory': 256,
        'mips': 1000,
        'cost': 0.10,
        'maxFreq': 1.0,
        'minFreq': 0.50,
        'minVoltage': 5.0,
        'maxVoltage': 7.0,
        'lambda': 0.000150
    },
    'Type_2': {
        'Task_Memory': 512,
        'mips': 1500,
        'cost': 0.20,
        'maxFreq': 1.5,
        'minFreq': 0.75,
        'minVoltage': 7.0,
        'maxVoltage': 9.0,
        'lambda': 0.000100
    },
    'Type_3': {
        'Task_Memory': 1024,
        'mips': 2000,
        'cost': 0.32,
        'maxFreq': 2.0,
        'minFreq': 1.00,
        'minVoltage': 9.0,
        'maxVoltage': 11.0,
        'lambda': 0.000080
    },
    'Type_4': {
        'Task_Memory': 500,
        'mips': 2500,
        'cost': 0.46,
        'maxFreq': 2.5,
        'minFreq': 1.25,
        'minVoltage': 11.0,
        'maxVoltage': 13.0,
        'lambda': 0.000045
    },
    'Type_5': {
        'Task_Memory': 560,
        'mips': 3000,
        'cost': 0.58,
        'maxFreq': 3.0,
        'minFreq': 1.50,
        'minVoltage': 13,
        'maxVoltage': 15,
        'lambda': 0.000040
    },
    'Type_6': {
        'Task_Memory': 250,
        'mips': 3500,
        'cost': 0.73,
        'maxFreq': 3.5,
        'minFreq': 1.75,
        'minVoltage': 15,
        'maxVoltage': 17,
        'lambda': 0.000025
    },
    'Type_7': {
        'Task_Memory': 150,
        'mips': 4000,
        'cost': 0.90,
        'maxFreq': 4.0,
        'minFreq': 2.00,
        'minVoltage': 17,
        'maxVoltage': 19,
        'lambda': 0.000010
    },
    'Type_8': {
        'Task_Memory': 750,
        'mips': 4500,
        'cost': 1.05,
        'maxFreq': 4.5,
        'minFreq': 2.25,
        'minVoltage': 19,
        'maxVoltage': 21,
        'lambda': 0.000005
    },
    'Type_9': {
        'Task_Memory': 2048,
        'mips': 500,
        'cost': 1.40,
        'maxFreq': 5.5,
        'minFreq': 2.75,
        'minVoltage': 23,
        'maxVoltage': 25,
        'lambda': 0.000001
    }
}


class VM:
    def __init__(self, name, Host, Type):
        self.name = name
        self.Host = Host
        self.Type = Type
        self.network = None

    def attach_network(self, network_config):
        self.network = network_config


def check_memory(datacenter, host_list):
    datacenter_memory = datacenter.memory
    host_memory = 0

    for host in host_list:
        host_memory += host.memory

    if host_memory > datacenter_memory:
        return True

    return False


def create_suitable_vms(task_memory, vm_memory):
    suitable_vms = []

    for t in task_memory:
        suitable_vms_for_task = []

        for v in range(len(vm_memory)):
            if t <= vm_memory[v]:
                suitable_vms_for_task.append(v)

        suitable_vms.append(suitable_vms_for_task)

    return suitable_vms


class VMAllocationEnv:

    def __init__(self, no_of_tasks, no_of_vms, task_lengths):
        super().__init__()

        self.task_lengths = task_lengths
        self.no_of_tasks = no_of_tasks
        self.no_of_vms = no_of_vms

        self.observation_space = spaces.Box(
            low=0,
            high=1,
            shape=(no_of_vms, no_of_tasks),
            dtype=np.float32
        )

        self.action_space = spaces.MultiDiscrete([no_of_vms, no_of_tasks])

        self.state = np.zeros(
            (no_of_vms, no_of_tasks),
            dtype=int
        )

    def calculate_reward(self, state, action, task_lengths):
        next_state = self.get_next_state(state, action)
        makespan = self.calculate_makespan(next_state, task_lengths)

        return 1.0 / float(makespan)

    def get_next_state(self, state, action):
        vm_idx = action[0]
        task_idx = action[1]

        next_state = state.copy()

        for i in range(len(next_state)):
            if next_state[i][task_idx] == 1:
                next_state[i][task_idx] = 0

        next_state[vm_idx][task_idx] = 1

        return next_state

    def calculate_makespan(self, state, task_lengths):
        makespan = 0.0

        for i in range(len(state)):
            time_in_row = 0.0

            for j in range(len(state[0])):
                if state[i][j] == 1:
                    time_in_row += task_lengths[j]

            makespan = max(makespan, time_in_row)

        return makespan

    def are_all_assigned(self, state):
        f = 0

        for i in range(len(state[0])):
            for j in range(len(state)):
                if state[j][i] == 1:
                    f += 1

        if f == self.no_of_tasks:
            return True

        return False

    def step(self, action):
        reward = self.calculate_reward(
            self.state,
            action,
            self.task_lengths
        )

        self.state = self.get_next_state(
            self.state,
            action
        )

        done = self.are_all_assigned(self.state)
        truncated = False
        info = {}

        return self.state, reward, done, info

    def reset(self):
        self.state = np.zeros(
            (self.no_of_vms, self.no_of_tasks),
            dtype=int
        )

        return self.state


# Create simulated datacenter
D2 = Datacenter(
    "D2",
    "Bangalore",
    200000,
    7.5,
    7.5,
    300
)


# Create simulated hosts
host_list = [
    Host("host1", D2, 2, 1024, 2, 1, 10),
    Host("host2", D2, 4, 2048, 4, 2, 25),
    Host("host3", D2, 8, 512, 4, 3, 100),
    Host("host4", D2, 16, 4096, 8, 4, 75)
]


# Check datacenter memory
if check_memory(D2, host_list):
    print("Memory limit Exceeded")


# Create simulated VMs
vm_list = [
    VM("VM1", "host1", "Type_1"),
    VM("VM2", "host1", "Type_2"),
    VM("VM3", "host2", "Type_3"),
    VM("VM4", "host2", "Type_1"),
    VM("VM5", "host3", "Type_2"),
    VM("VM6", "host3", "Type_3"),
    VM("VM7", "host4", "Type_1"),
    VM("VM8", "host4", "Type_2"),
    VM("VM9", "host1", "Type_1")
]


# ---------------------------------------------------------
# Connect the provided workload dataset to the simulation
# ---------------------------------------------------------

# Use the first 100 Run Time values from dataset.xlsx
task_lengths = RunTime[:100]

no_of_vms = len(vm_list)
no_of_tasks = len(task_lengths)


# Create the cloud scheduling environment
env = VMAllocationEnv(
    no_of_tasks,
    no_of_vms,
    task_lengths
)


# ---------------------------------------------------------
# Verification
# ---------------------------------------------------------

print("Cloud simulation environment created successfully")
print("Datacenter:", D2.name)
print("Location:", D2.location)
print("Number of hosts:", len(host_list))
print("Number of VMs:", no_of_vms)
print("Number of workload tasks:", no_of_tasks)
print("First 5 task lengths:", task_lengths[:5])
print("Observation space:", env.observation_space)
print("Action space:", env.action_space)

Cloud simulation environment created successfully
Datacenter: D2
Location: Bangalore
Number of hosts: 4
Number of VMs: 9
Number of workload tasks: 100
First 5 task lengths: [14400 14400    45   154    50]
Observation space: Box(0.0, 1.0, (9, 100), float32)
Action space: MultiDiscrete([  9 100])


In [ ]:
# ============================================================
# CLOUD NETWORK & SECURITY SIMULATION
# ============================================================

class NetworkConfig:
    def __init__(self, private_ip, public_ip=None):
        self.private_ip = private_ip
        self.public_ip = public_ip
        self.open_ports = {}
        self.firewall_rules = []

    def add_service(self, port, protocol, service):
        self.open_ports[port] = {
            "protocol": protocol,
            "service": service
        }

    def add_firewall_rule(self, port, source, action):
        self.firewall_rules.append({
            "port": port,
            "source": source,
            "action": action
        })


# Simulated network configuration for each VM
network_configs = {}

network_configs["VM1"] = NetworkConfig(
    private_ip="10.0.1.10",
    public_ip="203.0.113.10"
)

network_configs["VM2"] = NetworkConfig(
    private_ip="10.0.1.11",
    public_ip="203.0.113.11"
)

network_configs["VM3"] = NetworkConfig(
    private_ip="10.0.1.12"
)

network_configs["VM4"] = NetworkConfig(
    private_ip="10.0.1.13",
    public_ip="203.0.113.13"
)

network_configs["VM5"] = NetworkConfig(
    private_ip="10.0.1.14"
)

network_configs["VM6"] = NetworkConfig(
    private_ip="10.0.1.15",
    public_ip="203.0.113.15"
)

network_configs["VM7"] = NetworkConfig(
    private_ip="10.0.1.16"
)

network_configs["VM8"] = NetworkConfig(
    private_ip="10.0.1.17",
    public_ip="203.0.113.17"
)

network_configs["VM9"] = NetworkConfig(
    private_ip="10.0.1.18"
)


# ============================================================
# Simulated services / open ports
# ============================================================

# VM1 - Web + SSH
network_configs["VM1"].add_service(22, "TCP", "SSH")
network_configs["VM1"].add_service(80, "TCP", "HTTP")
network_configs["VM1"].add_service(443, "TCP", "HTTPS")

# VM2 - Web server
network_configs["VM2"].add_service(80, "TCP", "HTTP")
network_configs["VM2"].add_service(443, "TCP", "HTTPS")

# VM3 - Internal database
network_configs["VM3"].add_service(3306, "TCP", "MySQL")

# VM4 - SSH + application
network_configs["VM4"].add_service(22, "TCP", "SSH")
network_configs["VM4"].add_service(8080, "TCP", "Application")

# VM5 - Internal database
network_configs["VM5"].add_service(5432, "TCP", "PostgreSQL")

# VM6 - Web + SSH
network_configs["VM6"].add_service(22, "TCP", "SSH")
network_configs["VM6"].add_service(443, "TCP", "HTTPS")

# VM7 - Internal application
network_configs["VM7"].add_service(8080, "TCP", "Application")

# VM8 - Web server
network_configs["VM8"].add_service(80, "TCP", "HTTP")
network_configs["VM8"].add_service(443, "TCP", "HTTPS")

# VM9 - SSH
network_configs["VM9"].add_service(22, "TCP", "SSH")


# ============================================================
# Simulated firewall / security-group rules
# ============================================================

# VM1
network_configs["VM1"].add_firewall_rule(22, "0.0.0.0/0", "ALLOW")
network_configs["VM1"].add_firewall_rule(80, "0.0.0.0/0", "ALLOW")
network_configs["VM1"].add_firewall_rule(443, "0.0.0.0/0", "ALLOW")

# VM2
network_configs["VM2"].add_firewall_rule(80, "0.0.0.0/0", "ALLOW")
network_configs["VM2"].add_firewall_rule(443, "0.0.0.0/0", "ALLOW")

# VM3 - database restricted to internal network
network_configs["VM3"].add_firewall_rule(3306, "10.0.0.0/24", "ALLOW")

# VM4
network_configs["VM4"].add_firewall_rule(22, "192.168.1.0/24", "ALLOW")
network_configs["VM4"].add_firewall_rule(8080, "0.0.0.0/0", "ALLOW")

# VM5 - database restricted to internal network
network_configs["VM5"].add_firewall_rule(5432, "10.0.0.0/24", "ALLOW")

# VM6
network_configs["VM6"].add_firewall_rule(22, "0.0.0.0/0", "ALLOW")
network_configs["VM6"].add_firewall_rule(443, "0.0.0.0/0", "ALLOW")

# VM7 - internal application
network_configs["VM7"].add_firewall_rule(8080, "10.0.0.0/24", "ALLOW")

# VM8
network_configs["VM8"].add_firewall_rule(80, "0.0.0.0/0", "ALLOW")
network_configs["VM8"].add_firewall_rule(443, "0.0.0.0/0", "ALLOW")

# VM9
network_configs["VM9"].add_firewall_rule(22, "192.168.1.0/24", "ALLOW")


# ============================================================
# Verify the simulated network environment
# ============================================================

print("Cloud network simulation created successfully")
print("=" * 60)

for vm_name, config in network_configs.items():

    exposure = "PUBLIC" if config.public_ip else "PRIVATE"

    print(f"\n{vm_name}")
    print(f"  Private IP : {config.private_ip}")
    print(f"  Public IP  : {config.public_ip}")
    print(f"  Exposure   : {exposure}")

    print("  Services:")

    for port, service in config.open_ports.items():
        print(
            f"    {port}/"
            f"{service['protocol']} - "
            f"{service['service']}"
        )

    print("  Firewall rules:")

    for rule in config.firewall_rules:
        print(
            f"    Port {rule['port']} | "
            f"Source {rule['source']} | "
            f"{rule['action']}"
        )

Cloud network simulation created successfully

VM1
  Private IP : 10.0.1.10
  Public IP  : 203.0.113.10
  Exposure   : PUBLIC
  Services:
    22/TCP - SSH
    80/TCP - HTTP
    443/TCP - HTTPS
  Firewall rules:
    Port 22 | Source 0.0.0.0/0 | ALLOW
    Port 80 | Source 0.0.0.0/0 | ALLOW
    Port 443 | Source 0.0.0.0/0 | ALLOW

VM2
  Private IP : 10.0.1.11
  Public IP  : 203.0.113.11
  Exposure   : PUBLIC
  Services:
    80/TCP - HTTP
    443/TCP - HTTPS
  Firewall rules:
    Port 80 | Source 0.0.0.0/0 | ALLOW
    Port 443 | Source 0.0.0.0/0 | ALLOW

VM3
  Private IP : 10.0.1.12
  Public IP  : None
  Exposure   : PRIVATE
  Services:
    3306/TCP - MySQL
  Firewall rules:
    Port 3306 | Source 10.0.0.0/24 | ALLOW

VM4
  Private IP : 10.0.1.13
  Public IP  : 203.0.113.13
  Exposure   : PUBLIC
  Services:
    22/TCP - SSH
    8080/TCP - Application
  Firewall rules:
    Port 22 | Source 192.168.1.0/24 | ALLOW
    Port 8080 | Source 0.0.0.0/0 | ALLOW

VM5
  Private IP : 10.0.1.14
  Public

In [ ]:
# ============================================================
# Attach network configuration to simulated VMs
# ============================================================

for vm in vm_list:
    if vm.name in network_configs:
        vm.attach_network(network_configs[vm.name])

print("Network configurations attached to VMs")
print("=" * 60)

for vm in vm_list:
    print(
        f"{vm.name} | "
        f"Host: {vm.Host} | "
        f"Type: {vm.Type} | "
        f"Private IP: {vm.network.private_ip}"
    )

Network configurations attached to VMs
VM1 | Host: host1 | Type: Type_1 | Private IP: 10.0.1.10
VM2 | Host: host1 | Type: Type_2 | Private IP: 10.0.1.11
VM3 | Host: host2 | Type: Type_3 | Private IP: 10.0.1.12
VM4 | Host: host2 | Type: Type_1 | Private IP: 10.0.1.13
VM5 | Host: host3 | Type: Type_2 | Private IP: 10.0.1.14
VM6 | Host: host3 | Type: Type_3 | Private IP: 10.0.1.15
VM7 | Host: host4 | Type: Type_1 | Private IP: 10.0.1.16
VM8 | Host: host4 | Type: Type_2 | Private IP: 10.0.1.17
VM9 | Host: host1 | Type: Type_1 | Private IP: 10.0.1.18


In [ ]:
# ============================================================
# Fix: Attach existing host_list to Datacenter D2
# ============================================================

D2.hosts = host_list

print("Hosts attached to datacenter successfully")
print("=" * 60)

print(f"Datacenter: {D2.name}")
print(f"Location: {D2.location}")
print(f"Number of hosts: {len(D2.hosts)}")

for host in D2.hosts:
    print(
        f"{host.name} | "
        f"CPU Cores: {host.cpu_cores} | "
        f"Memory: {host.memory} | "
        f"RAM: {host.ram} | "
        f"Bandwidth: {host.bandwidth}"
    )

Hosts attached to datacenter successfully
Datacenter: D2
Location: Bangalore
Number of hosts: 4
host1 | CPU Cores: 2 | Memory: 1024 | RAM: 2 | Bandwidth: 10
host2 | CPU Cores: 4 | Memory: 2048 | RAM: 4 | Bandwidth: 25
host3 | CPU Cores: 8 | Memory: 512 | RAM: 4 | Bandwidth: 100
host4 | CPU Cores: 16 | Memory: 4096 | RAM: 8 | Bandwidth: 75


In [ ]:
# ============================================================
# Milestone 2 - Environment Resource Summary
# ============================================================

print("=" * 70)
print("MILESTONE 2 - CLOUD & NETWORK ENVIRONMENT SUMMARY")
print("=" * 70)

print(f"\nDatacenter      : {D2.name}")
print(f"Location        : {D2.location}")
print(f"Total Hosts     : {len(D2.hosts)}")
print(f"Total VMs       : {len(vm_list)}")
print(f"Workload Tasks  : {no_of_tasks}")

print("\n" + "-" * 70)
print("HOST RESOURCES")
print("-" * 70)

for host in D2.hosts:
    print(
        f"{host.name} | "
        f"CPU Cores: {host.cpu_cores} | "
        f"Memory: {host.memory} | "
        f"RAM: {host.ram} | "
        f"Bandwidth: {host.bandwidth}"
    )

print("\n" + "-" * 70)
print("VIRTUAL MACHINES & NETWORK RESOURCES")
print("-" * 70)

for vm in vm_list:
    net = vm.network

    public_status = (
        f"Public IP: {net.public_ip}"
        if net.public_ip
        else "Private Only"
    )

    print(
        f"{vm.name} | "
        f"Host: {vm.Host} | "
        f"Type: {vm.Type} | "
        f"Private IP: {net.private_ip} | "
        f"{public_status} | "
        f"Open Ports: {list(net.open_ports.keys())}"
    )

print("\n" + "-" * 70)
print("WORKLOAD DATASET")
print("-" * 70)

print(f"Dataset rows       : {len(df)}")
print(f"Dataset columns    : {len(df.columns)}")
print(f"Workload variable  : Run Time")
print(f"Tasks used in env  : {no_of_tasks}")
print(f"Sample task times  : {task_lengths[:5]}")

print("\n" + "=" * 70)
print("Milestone 2 environment setup completed successfully.")
print("=" * 70)

MILESTONE 2 - CLOUD & NETWORK ENVIRONMENT SUMMARY

Datacenter      : D2
Location        : Bangalore
Total Hosts     : 4
Total VMs       : 9
Workload Tasks  : 100

----------------------------------------------------------------------
HOST RESOURCES
----------------------------------------------------------------------
host1 | CPU Cores: 2 | Memory: 1024 | RAM: 2 | Bandwidth: 10
host2 | CPU Cores: 4 | Memory: 2048 | RAM: 4 | Bandwidth: 25
host3 | CPU Cores: 8 | Memory: 512 | RAM: 4 | Bandwidth: 100
host4 | CPU Cores: 16 | Memory: 4096 | RAM: 8 | Bandwidth: 75

----------------------------------------------------------------------
VIRTUAL MACHINES & NETWORK RESOURCES
----------------------------------------------------------------------
VM1 | Host: host1 | Type: Type_1 | Private IP: 10.0.1.10 | Public IP: 203.0.113.10 | Open Ports: [22, 80, 443]
VM2 | Host: host1 | Type: Type_2 | Private IP: 10.0.1.11 | Public IP: 203.0.113.11 | Open Ports: [80, 443]
VM3 | Host: host2 | Type: Type_3 | Pr

In [ ]:
# ============================================================
# Milestone 2 - Resource Identification
# ============================================================

print("=" * 70)
print("RESOURCE IDENTIFICATION")
print("=" * 70)

# -------------------------
# Cloud Resources
# -------------------------

print("\nCLOUD RESOURCES")
print("-" * 70)

print(f"Datacenter : {D2.name}")
print(f"Location   : {D2.location}")
print(f"Hosts      : {len(D2.hosts)}")
print(f"VMs        : {len(vm_list)}")

# -------------------------
# Host Resources
# -------------------------

print("\nHOST RESOURCE INVENTORY")
print("-" * 70)

for host in D2.hosts:
    print(
        f"{host.name} | "
        f"CPU Cores={host.cpu_cores} | "
        f"Memory={host.memory} | "
        f"RAM={host.ram} | "
        f"Bandwidth={host.bandwidth}"
    )

# -------------------------
# VM Resources
# -------------------------

print("\nVM RESOURCE INVENTORY")
print("-" * 70)

for vm in vm_list:
    vm_type = VM_type[vm.Type]

    print(
        f"{vm.name} | "
        f"Host={vm.Host} | "
        f"Type={vm.Type} | "
        f"Task Memory={vm_type['Task_Memory']} | "
        f"MIPS={vm_type['mips']}"
    )

# -------------------------
# Network Resources
# -------------------------

print("\nNETWORK RESOURCE INVENTORY")
print("-" * 70)

total_open_ports = 0
public_vms = 0

for vm in vm_list:
    net = vm.network

    if net.public_ip:
        public_vms += 1

    total_open_ports += len(net.open_ports)

    services = ", ".join(
        f"{port}/{data['protocol']} ({data['service']})"
        for port, data in net.open_ports.items()
    )

    print(
        f"{vm.name} | "
        f"Private IP={net.private_ip} | "
        f"Public IP={net.public_ip if net.public_ip else 'None'} | "
        f"Services={services}"
    )

print("\n" + "-" * 70)
print("NETWORK SUMMARY")
print("-" * 70)

print(f"VMs with simulated public exposure : {public_vms}")
print(f"Total configured open ports        : {total_open_ports}")

# -------------------------
# Workload Resources
# -------------------------

print("\nWORKLOAD RESOURCE")
print("-" * 70)

print(f"Dataset records : {len(df)}")
print(f"Dataset fields  : {len(df.columns)}")
print(f"Workload field  : Run Time")
print(f"Tasks simulated : {no_of_tasks}")

print("\n" + "=" * 70)
print("RESOURCE IDENTIFICATION COMPLETED")
print("=" * 70)

RESOURCE IDENTIFICATION

CLOUD RESOURCES
----------------------------------------------------------------------
Datacenter : D2
Location   : Bangalore
Hosts      : 4
VMs        : 9

HOST RESOURCE INVENTORY
----------------------------------------------------------------------
host1 | CPU Cores=2 | Memory=1024 | RAM=2 | Bandwidth=10
host2 | CPU Cores=4 | Memory=2048 | RAM=4 | Bandwidth=25
host3 | CPU Cores=8 | Memory=512 | RAM=4 | Bandwidth=100
host4 | CPU Cores=16 | Memory=4096 | RAM=8 | Bandwidth=75

VM RESOURCE INVENTORY
----------------------------------------------------------------------
VM1 | Host=host1 | Type=Type_1 | Task Memory=256 | MIPS=1000
VM2 | Host=host1 | Type=Type_2 | Task Memory=512 | MIPS=1500
VM3 | Host=host2 | Type=Type_3 | Task Memory=1024 | MIPS=2000
VM4 | Host=host2 | Type=Type_1 | Task Memory=256 | MIPS=1000
VM5 | Host=host3 | Type=Type_2 | Task Memory=512 | MIPS=1500
VM6 | Host=host3 | Type=Type_3 | Task Memory=1024 | MIPS=2000
VM7 | Host=host4 | Type=Type_1 |

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
class PolicyNet(nn.Module):

    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(no_of_tasks, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc_mu = nn.Linear(64, 2)

    def forward(self, x):
        x = torch.tensor(x).float().to(device)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        loc = self.fc_mu(x)
        loc = torch.tanh(loc*9)
        return loc
def Actor(states):
    actor = PolicyNet()
    actor = actor.to(device)
    actor_ = actor.to(device)
    actor_.requires_grad_(False)
    actions = actor(states).numpy()[0]
    action = [abs(round(actions[0]*9.45)), abs(round(actions[1]*60.325))]
    return action

actor = PolicyNet()
actor = actor.to(device)
actor.requires_grad_(False)

PolicyNet(
  (fc1): Linear(in_features=100, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=64, bias=True)
  (fc_mu): Linear(in_features=64, out_features=2, bias=True)
)

In [ ]:
class ValueNet(nn.Module):

    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(no_of_tasks, 64)
        self.fc_mu = nn.Linear(64, 2)

    def forward(self, x):
        x = torch.tensor(x).float().to(device)
        x = F.relu(self.fc1(x))
        loc = self.fc_mu(x)
        loc = torch.tanh(loc*9)
        return loc
def Critic(states):
    critic = ValueNet()
    critic = critic.to(device)
    critic_ = critic.to(device)
    critic_.requires_grad_(False)
    actions = critic(states).numpy()[0]
    action = [abs(round(actions[0]*9.45)), abs(round(actions[1]*60.325))]
    return action

critic = ValueNet()
critic = critic.to(device)
critic.requires_grad_(False)

ValueNet(
  (fc1): Linear(in_features=100, out_features=64, bias=True)
  (fc_mu): Linear(in_features=64, out_features=2, bias=True)
)

In [ ]:
num_envs = mp.cpu_count()
num_envs

2

In [ ]:
alpha=1e-4
gamma=0.99
reward_records = []
States = []
Final_State = []
actor_loss=[]
critic_loss=[]
final_makespan = []
Task_Allocation = {}
episodes=[]
actor_optim = AdamW(actor.parameters(), lr=1e-3)
critic_optim = AdamW(critic.parameters(), lr=1e-3)
stats = {'Actor Loss': [], 'Critic Loss': [], 'Returns': []}
advantage = 0
warnings.filterwarnings('ignore')
for i in tqdm(range(5)):
    states = torch.from_numpy(env.reset()).float()
    done = False
    cum_reward = 0
    makespans =[]
    ep_return = torch.zeros((num_envs, 1))
    I = 1.

    episodes.append(i)

    while not done:
        # Taking Action Based on Advantage
        Action = actor(states)[0]
        if advantage<=0:
            action = Actor(states)
        else:
            action = Critic(states)
        # if action[0]>=10 or action[1]>=60:
        #     continue
        a = env.action_space.sample()
        next_state, reward, done, _ = env.step(a)
        if done:
#             print('done!')
            makespans.append(1/reward)
            print("Makespans: ", makespans)
            States.append(states)
            cum_reward += reward
            break

#         Critic loss
        temp =  critic(torch.from_numpy(next_state).float())[:][0]
        value = temp
        target = reward + gamma * temp
        critic_loss = F.mse_loss(value, target)
        critic.zero_grad()
        critic_optim.step()

#         Advantage Calculation
        value = sum(value/10)
        target = sum(target/10)
#         value = math.sqrt((value[0]**2+value[1]**2)/2)
#         target = math.sqrt((target[0]**2+target[1]**2)/2)
        advantage = (target - value)

#         Actor loss and returns
        probs = actor(states)[1]
        log_probs = torch.log(probs + 1e-6)
        entropy = - torch.sum(probs * log_probs, dim=-1, keepdim=True)
        actor_loss = - I * log_probs * advantage - 0.01 * entropy
        actor_loss = actor_loss.mean()
        actor.zero_grad()
        actor_optim.step()
        ep_return += reward
        states = next_state
        I = I * gamma

    reward_records.append(cum_reward)
    print("Completed episode  ",i, " with reward:  ", cum_reward, " and Makespan: ", makespans)

    final_makespan.append(min(makespans))
    Final_State.append(States[final_makespan.index(min(makespans))])
    stats['Actor Loss'].append(actor_loss.item())
    stats['Critic Loss'].append(critic_loss.item())
    stats['Returns'].append(ep_return.mean().item())

print("Final Makespan : ", min(final_makespan))
print("\n Done")

 20%|██        | 1/5 [00:04<00:18,  4.66s/it]

Makespans:  [270056.0]
Completed episode   0  with reward:   3.702935687412981e-06  and Makespan:  [270056.0]


 40%|████      | 2/5 [00:06<00:08,  2.86s/it]

Makespans:  [446285.99999999994]
Completed episode   1  with reward:   2.240715594932398e-06  and Makespan:  [446285.99999999994]


 60%|██████    | 3/5 [00:07<00:04,  2.15s/it]

Makespans:  [474879.99999999994]
Completed episode   2  with reward:   2.1057951482479786e-06  and Makespan:  [474879.99999999994]


 80%|████████  | 4/5 [00:08<00:01,  1.85s/it]

Makespans:  [318750.0]
Completed episode   3  with reward:   3.137254901960784e-06  and Makespan:  [318750.0]


100%|██████████| 5/5 [00:10<00:00,  2.02s/it]

Makespans:  [307481.0]
Completed episode   4  with reward:   3.25223347133644e-06  and Makespan:  [307481.0]
Final Makespan :  270056.0

 Done


In [ ]:
Final_VM = Final_State[final_makespan.index(min(final_makespan))]
print(Final_VM)
for i in range(no_of_vms):
    Temp_List = []
    for j in range(no_of_tasks):
        if Final_VM[i][j]==1:
            Temp_List.append(j)
    Task_Allocation[i] = Temp_List
final_makespan.index(min(final_makespan))

[[1 0 0 0 0 0 0 0 1 0 1 0 0 0 0 1 0 0 0 0 0 0 0 0 1 1 0 0 0 1 0 0 0 1 0 0
  1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 1 1 0 0 0 0 0 1 0 0 0 0
  1 0 0 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 1 0 0]
 [0 1 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0
  0 0 0 0 0 1 0 0 0 0 1 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 0 1 0 0 0 0 0
  0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 1 0
  0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 1 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1
  0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 1 1 0 0 0 0 0 0 0 0 1 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 

0

In [ ]:
print("Minimum Time required to Schedule all task are:",min(final_makespan))
print("\nTask Allocation on VM's :")
for key,val in Task_Allocation.items():
    print(str(key)+' : '+str(val))

Minimum Time required to Schedule all task are: 270056.0

Task Allocation on VM's :
0 : [0, 8, 10, 15, 24, 25, 29, 33, 36, 56, 60, 61, 67, 72, 83, 85, 97]
1 : [1, 11, 32, 51, 55, 68, 77, 82, 88, 99]
2 : [12, 26, 27, 28, 30, 44, 58, 59, 70, 76, 84, 91, 95]
3 : [34, 50, 54, 71, 80, 81, 92]
4 : [14, 16, 21, 38, 39, 48, 53, 65]
5 : [13, 20, 22, 23, 35, 43, 46, 52, 62, 69, 96, 98]
6 : [3, 5, 18, 19, 45, 47, 57, 63, 79, 86, 87, 90]
7 : [2, 6, 17, 31, 37, 40, 64, 66, 73, 78, 89, 93, 94]
8 : [7, 9, 41, 42, 49, 74, 75]
